In [1]:
import chromadb
from sentence_transformers import SentenceTransformer
import pickle
import re

/home/kxelina/RAG_project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Initialize embedding model 
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dim = embedding_model.get_sentence_embedding_dimension()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8526.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Initialize ChromaDB with two collections for different retrieval patterns
# Delete existing collections for clean rebuild
client = chromadb.PersistentClient(path="./chroma_db")
for col in ["global_summaries", "grouped_summaries"]:
    try: client.delete_collection(name=col)
    except: pass

# Create collections with cosine similarity for ranking
collection_global = client.get_or_create_collection(name="global_summaries", metadata={"hnsw:space": "cosine"})
collection_grouped = client.get_or_create_collection(name="grouped_summaries", metadata={"hnsw:space": "cosine"})

In [4]:
# Load preprocessed data
with open("analysis.pkl", "rb") as f:
    data = pickle.load(f)

global_docs = data['global']['documents']
grouped_docs = data['grouped']['documents']

print(f"  Global: {len(global_docs)} chunks")
print(f"  Grouped: {len(grouped_docs)} chunks")

  Global: 54 chunks
  Grouped: 1312 chunks


In [ ]:
# Helper function to add documents to collection
# Batch processing for efficiency
def add_to_collection(collection, documents, collection_name, batch_size=50):
    total = len(documents)
    
    for i in range(0, total, batch_size):
        batch = documents[i:i + batch_size]
        
        # Create unique IDs combining collection name and chunk ID
        ids = [f"{collection_name}_{doc.metadata['chunk_id']}" for doc in batch]
        texts = [doc.page_content for doc in batch]
        metadatas = [doc.metadata for doc in batch]
        
        # Generate embeddings for semantic search
        embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()
        
        # Store documents with embeddings and metadata
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )

In [6]:
# Add layers to their respective collections
add_to_collection(collection_global, global_docs, "global")
add_to_collection(collection_grouped, grouped_docs, "grouped")

In [ ]:
# Extract functions - parse formatted text for sorting/ranking results
def extract_amount(text):
    # Parse dollar amounts for default sorting
    match = re.search(r'\$[\d,]+\.?\d*', text)
    return float(match.group().replace('$', '').replace(',', '')) if match else 0

def extract_margin(text):
    # Parse profit margins for profitability rankings
    match = re.search(r'Margin\s+([-\d.]+)%', text)
    return float(match.group(1)) if match else 0

def extract_discount(text):
    # Parse discount percentages for discount-related queries
    match = re.search(r'Average discount (\d+)%', text)
    return float(match.group(1)) if match else 0

In [ ]:
def retrieve_context(query, num_results=20):
    # Special handling for discount queries - pull from grouped collection
    if "discount" in query.lower() or "frequently sold" in query.lower():
        # Similarity search: Query the grouped collection for relevant documents based on semantic similarity
        grouped_results = collection_grouped.query(
            query_texts=["frequently discounted"],
            n_results=2000,
            include=["documents", "metadatas"])

        documents = []
        # Metadata filtering: Filter only documents containing "Frequently discounted product" to focus on relevant insights
        if grouped_results['documents'] and len(grouped_results['documents']) > 0:
            for doc in grouped_results['documents'][0]:
                if "Frequently discounted product" in doc:
                    documents.append(doc)

        documents = sorted(documents, key=extract_discount, reverse=True)
        documents = documents[:num_results]
        context = "\n".join(documents)
        return context[:2500] if len(context) > 2500 else context

    # Similarity search for other queries - pull from both collections and combine results
    global_results = collection_global.query(
        query_texts=[query],
        n_results=num_results,
        include=["documents", "metadatas"]
    )

    grouped_results = collection_grouped.query(
        query_texts=[query],
        n_results=num_results,
        include=["documents", "metadatas"])

    documents = []

    # Metadata filtering: Only include aggregate metrics from global layer
    if global_results['documents'] and len(global_results['documents']) > 0:
        for doc, meta in zip(global_results['documents'][0], global_results['metadatas'][0]):
            if not meta.get('is_aggregate', True):
                continue
            documents.append(doc)

    # Metadata filtering: Include all grouped summaries
    if grouped_results['documents'] and len(grouped_results['documents']) > 0:
        for doc, meta in zip(grouped_results['documents'][0], grouped_results['metadatas'][0]):
            documents.append(doc)

    # Determine sort key based on query type
    if "margin" in query.lower() and ("sub-categ" in query.lower() or "category" in query.lower()):
        documents = sorted(documents, key=extract_margin, reverse=True)
    else:
        documents = sorted(documents, key=extract_amount, reverse=True)

    documents = documents[:num_results]
    context = "\n".join(documents)
    return context[:2500] if len(context) > 2500 else context

In [ ]:
# Test queries with expected answers for validation
# Run these to verify vector DB retrieval accuracy before RAG integration
test_queries = [
    ("What is the sales trend over the 4-year period?", 
     "EXPECTED: 2014: $484,247.50, 2015: $470,532.51, 2016: $609,205.60, 2017: $733,215.26 (51.5% growth)"),
    
    ("Which months show the highest sales? Is there seasonality?", 
     "EXPECTED: November ($352,461.07), December ($325,293.50), September ($307,649.95) - Peak in Nov-Dec"),
    
    ("How has profit margin changed over time?", 
     "EXPECTED: 2014: 11.81%, 2015: 11.76%, 2016: 12.98%, 2017: 11.60%"),
    
    ("Which product category generates the most revenue?", 
     "EXPECTED: Technology ($836,154.03, 36.4%), Furniture ($741,999.80, 32.3%), Office Supplies ($719,047.03, 31.3%)"),
    
    ("What sub-categories have the highest profit margins?", 
     "EXPECTED: Labels (44.42%), Paper (43.39%), Envelopes (42.27%), Copiers (37.20%), Fasteners (31.40%)"),
    
    ("Which products are frequently sold at a discount?", 
     "EXPECTED: Acco 6 Outlet Guardian Premium Plus Surge Suppressor (80%), Belkin F9S820V06 8 Outlet Surge (80%), Acco 6 Outlet Guardian Basic Surge Suppressor (80%)"),
    
    ("Which region has the best sales performance?", 
     "EXPECTED: West ($725,457.82), East ($678,781.24), Central ($501,239.89), South ($391,721.91)"),
    
    ("Compare sales performance across different states.", 
     "EXPECTED: California ($457,687.63), New York ($310,876.27), Texas ($170,188.05), Washington ($138,641.27), Pennsylvania ($116,511.91)"),
    
    ("Which cities are the top performers?", 
     "EXPECTED: New York City ($256,368.16), Los Angeles ($175,851.34), Seattle ($119,540.74), San Francisco ($112,669.09), Philadelphia ($109,077.01)"),
    
    ("Compare Technology vs Furniture sales trends.", 
     "EXPECTED: Technology ($836,154.03) > Furniture ($741,999.80), Technology leading"),
    
    ("How does the West region compare to the East in terms of profit?", 
     "EXPECTED: West ($108,418.45, 14.94% margin) > East ($91,522.78, 13.48% margin)")
]

for i, (query, expected) in enumerate(test_queries, 1):
    print(f"\n{i}. {query}")
    print(f"   {expected}")
    print("-" * 70)
    context = retrieve_context(query, num_results=20)
    print(context)
    print()


1. What is the sales trend over the 4-year period?
   EXPECTED: 2014: $484,247.50, 2015: $470,532.51, 2016: $609,205.60, 2017: $733,215.26 (51.5% growth)
----------------------------------------------------------------------
TOTAL AGGREGATE: Sales $2,297,200.86, Profit $286,397.02, Orders 9,994, Margin 12.47%
Category Technology: Sales $836,154.03, Profit $145,454.95, Margin 17.40%
Year 2017 TOTAL: Sales $733,215.26, Profit $93,439.27, Margin 11.60%
Region West: Sales $725,457.82, Profit $108,418.45, Margin 14.94%
Category Office Supplies: Sales $719,047.03, Profit $122,490.80, Margin 17.04%
Region East: Sales $678,781.24, Profit $91,522.78, Margin 13.48%
Year 2016 TOTAL: Sales $609,205.60, Profit $81,795.17, Margin 12.98%
Year 2014 TOTAL: Sales $484,247.50, Profit $49,543.97, Margin 11.81%
Year 2015 TOTAL: Sales $470,532.51, Profit $61,618.60, Margin 11.76%
State California: Sales $457,687.63, Profit $76,381.39, Margin 16.69%
Region South: Sales $391,721.91, Profit $46,749.43, Margin

In [ ]:
# Final check - verify total chunks loaded into vector DB
total = collection_global.count() + collection_grouped.count()
print(f"Vector DB: {total} total chunks")

Vector DB: 1366 total chunks
